In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [2]:
spark = (
    SparkSession.builder
    .appName("MovieRecommendationSystem")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/15 09:26:00 WARN Utils: Your hostname, DESKTOP-6I983CD, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/15 09:26:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 09:26:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
DATA_PATH = "../data/raw/ml-latest"

ratings = spark.read.csv(
    f"{DATA_PATH}/ratings.csv",
    header=True,
    inferSchema=True
)

movies = spark.read.csv(
    f"{DATA_PATH}/movies.csv",
    header=True,
    inferSchema=True
)

In [4]:
ratings.printSchema()
movies.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



In [5]:
print("Ratings:", ratings.count())
print("Movies:", movies.count())

Ratings: 33832162
Movies: 86537


In [6]:
ratings.show(5)

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|      1|   4.0|1225734739|
|     1|    110|   4.0|1225865086|
|     1|    158|   4.0|1225733503|
|     1|    260|   4.5|1225735204|
|     1|    356|   5.0|1225735119|
+------+-------+------+----------+
only showing top 5 rows


In [7]:
movies.show(5, truncate=False)

+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
+-------+----------------------------------+-------------------------------------------+
only showing top 5 rows


In [8]:
from pyspark.sql.functions import col, count, when

ratings.select(
    count(when(col("userId").isNull(), True)).alias("userId_nulls"),
    count(when(col("movieId").isNull(), True)).alias("movieId_nulls"),
    count(when(col("rating").isNull(), True)).alias("rating_nulls")
).show()

+------------+-------------+------------+
|userId_nulls|movieId_nulls|rating_nulls|
+------------+-------------+------------+
|           0|            0|           0|
+------------+-------------+------------+



In [9]:
ratings.columns

['userId', 'movieId', 'rating', 'timestamp']

In [10]:
movies.columns  

['movieId', 'title', 'genres']

In [11]:
print(ratings.select("userId").distinct().count())
print(movies.select("movieId").distinct().count())

330975
86537


In [12]:
ratings.groupBy("rating").count().orderBy("rating").show()

+------+-------+
|rating|  count|
+------+-------+
|   0.5| 566306|
|   1.0|1013645|
|   1.5| 562409|
|   2.0|2146492|
|   2.5|1760733|
|   3.0|6400664|
|   3.5|4465001|
|   4.0|8835955|
|   4.5|3123055|
|   5.0|4957902|
+------+-------+



In [13]:
from pyspark.sql.functions import count

movie_counts = (
    ratings.groupBy("movieId")
    .agg(count("*").alias("num_ratings"))
)

(
    movie_counts
    .join(movies, "movieId")
    .orderBy(movie_counts.num_ratings.desc())
    .select("title", "num_ratings")
    .show(10, truncate=False)
)

+-----------------------------------------------------+-----------+
|title                                                |num_ratings|
+-----------------------------------------------------+-----------+
|Shawshank Redemption, The (1994)                     |122296     |
|Forrest Gump (1994)                                  |113581     |
|Pulp Fiction (1994)                                  |108756     |
|Matrix, The (1999)                                   |107056     |
|Silence of the Lambs, The (1991)                     |101802     |
|Star Wars: Episode IV - A New Hope (1977)            |97202      |
|Fight Club (1999)                                    |86207      |
|Schindler's List (1993)                              |84232      |
|Jurassic Park (1993)                                 |83026      |
|Star Wars: Episode V - The Empire Strikes Back (1980)|80200      |
+-----------------------------------------------------+-----------+
only showing top 10 rows


In [14]:
ratings_sample = ratings.sample(fraction=0.05, seed=42)
print(ratings_sample.count())

1690038


In [15]:
train, test = ratings_sample.randomSplit([0.8, 0.2], seed=42)

In [16]:
print(train.count())
print(test.count())

1352262


337776


In [17]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True,
    rank=5,
    maxIter=5,
    regParam=0.1,
    seed=42
)

model = als.fit(train)

26/06/15 09:28:23 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [18]:
predictions = model.transform(test)

predictions.select(
    "userId",
    "movieId",
    "rating",
    "prediction"
).show(10)

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|    12|   1456|   1.0| 0.6016408|
|    14|     10|   4.5|    3.0615|
|    14|    110|   4.0| 3.1585379|
|    14|   1206|   4.5| 3.0900831|
|    14|   1732|   4.0|  3.255763|
|   148|    648|   2.0| 2.0462782|
|   148|   1285|   3.0|  2.565988|
|   161|   3094|   4.0| 2.7516184|
|   190|   7215|   4.5| 4.0388737|
|   198|    786|   2.0| 2.4866657|
+------+-------+------+----------+
only showing top 10 rows


In [19]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)

print(f"RMSE: {rmse}")

RMSE: 1.0691577489673425


In [20]:
user_recs = model.recommendForAllUsers(10)

In [21]:
user_recs.show(5, truncate=False)

+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                                                                                                                                  |
+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|12    |[{154672, 13.613564}, {166028, 12.607286}, {141375, 12.183456}, {116953, 12.063979}, {190309, 11.914094}, {110134, 11.700058}, {106897, 11.618612}, {176939, 11.612272}, {281662, 11.597651}, {99487, 11.421931}]|
|14    |[{166028, 11.267773}, {154672, 11.163468}, {212046, 10.848611}, {244216, 10.742166}, {116953, 9.89277}, {155758, 9.5

In [22]:
# Convert nested recommendations to rows 

from pyspark.sql.functions import explode, col 

user_recs_flat = (
    user_recs
    .withColumn("rec", explode("recommendations"))
    .select(
        col("userID"),
        col("rec.movieID").alias("movieID"),
        col("rec.rating").alias("predicted_rating")
    )
)

user_recs_flat.show(10)

+------+-------+----------------+
|userID|movieID|predicted_rating|
+------+-------+----------------+
|    12| 154672|       13.613564|
|    12| 166028|       12.607286|
|    12| 141375|       12.183456|
|    12| 116953|       12.063979|
|    12| 190309|       11.914094|
|    12| 110134|       11.700058|
|    12| 106897|       11.618612|
|    12| 176939|       11.612272|
|    12| 281662|       11.597651|
|    12|  99487|       11.421931|
+------+-------+----------------+
only showing top 10 rows


In [23]:
# Join movie titles 

recommendations_with_titles = (
    user_recs_flat
    .join(movies, on="movieID", how="left")
    .select("userID", "title", "genres", "predicted_rating")
)

recommendations_with_titles.show(20, truncate=False)

+------+-----------------------------------------------+-------------------------+----------------+
|userID|title                                          |genres                   |predicted_rating|
+------+-----------------------------------------------+-------------------------+----------------+
|12    |The Fourth Dimension (2012)                    |Drama                    |13.613564       |
|12    |What Remains of Us (2004)                      |Documentary              |12.607286       |
|12    |Sugarhouse (2007)                              |Drama|Thriller           |12.183456       |
|12    |Magic Camp (2012)                              |Children|Documentary     |12.063979       |
|12    |Re Loca (2018)                                 |Comedy                   |11.914094       |
|12    |Belle and Sebastien (Belle et Sébastien) (2013)|Adventure                |11.700058       |
|12    |Stuck Between Stations (2011)                  |Comedy|Drama|Romance     |11.618612       |


In [24]:
recommendations_with_titles.filter(col("userId") == 1).show(10,  truncate=False)


+------+-------------------------------+------------------------------+----------------+
|userID|title                          |genres                        |predicted_rating|
+------+-------------------------------+------------------------------+----------------+
|1     |What Remains of Us (2004)      |Documentary                   |7.196504        |
|1     |The Fourth Dimension (2012)    |Drama                         |6.933401        |
|1     |Irudhi Suttru (2016)           |Action|Drama                  |6.600689        |
|1     |The Secret (2002)              |Drama                         |6.4926777       |
|1     |The Last Full Measure (2020)   |Drama                         |6.4325213       |
|1     |Tender Scoundrel (1966)        |Comedy|Crime                  |6.2429194       |
|1     |Magic Camp (2012)              |Children|Documentary          |6.1441965       |
|1     |Sailor Moon R: The Movie (1993)|Animation|Comedy|Drama|Romance|5.9640207       |
|1     |Somebody is W

In [25]:
# Add popularity filter

movie_rating_counts = (
    ratings
    .groupBy("movieId")
    .agg(count("*").alias("num_ratings"))
)

In [26]:
recommendations_with_titles = (
    user_recs_flat
    .join(movies, on="movieId", how="left")
    .join(movie_rating_counts, on="movieId", how="left")
    .filter(col("num_ratings") >= 20)
    .select("userId", "title", "genres", "predicted_rating", "num_ratings")
)

In [27]:
recommendations_with_titles.filter(col("userId") == 1).show(10, truncate=False)

+------+-------------------------------+------------------------------+----------------+-----------+
|userId|title                          |genres                        |predicted_rating|num_ratings|
+------+-------------------------------+------------------------------+----------------+-----------+
|1     |Somebody is Waiting (1996)     |Drama                         |5.951395        |39         |
|1     |Sailor Moon R: The Movie (1993)|Animation|Comedy|Drama|Romance|5.9640207       |62         |
|1     |The Last Full Measure (2020)   |Drama                         |6.4325213       |48         |
+------+-------------------------------+------------------------------+----------------+-----------+



In [28]:
movies.filter(
    col("title").contains("Interstellar")
).show(truncate=False)

+-------+----------------------------------+-----------+
|movieId|title                             |genres     |
+-------+----------------------------------+-----------+
|109487 |Interstellar (2014)               |Sci-Fi|IMAX|
|204120 |The Science of Interstellar (2014)|Documentary|
+-------+----------------------------------+-----------+



In [29]:
interstaller_fans = ratings.filter( 
    (col("movieId") == 109487) & (col("rating") >= 4.5)
)

interstaller_fans.show(10)
print(interstaller_fans.count())

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     3| 109487|   5.0|1536174094|
|     4| 109487|   4.5|1442684878|
|    16| 109487|   4.5|1473468775|
|    30| 109487|   5.0|1566544733|
|    33| 109487|   4.5|1587180607|
|    37| 109487|   5.0|1578164943|
|    40| 109487|   4.5|1452041459|
|    49| 109487|   4.5|1487352952|
|    57| 109487|   5.0|1622601580|
|    87| 109487|   5.0|1442832717|
+------+-------+------+----------+
only showing top 10 rows


21038


In [30]:
fan_user_ids = interstaller_fans.select("userId").distinct()

similar_taste_movies = (
    ratings
    .join(fan_user_ids, on="userId")
    .filter(col("rating") >= 4.5)
    .join(movies, on="movieId")
    .groupBy("movieId", "title", "genres")
    .count()
    .orderBy(col("count").desc())
)

similar_taste_movies.show(20, truncate=False)

+-------+---------------------------------------------------------+-----------------------------------------------+-----+
|movieId|title                                                    |genres                                         |count|
+-------+---------------------------------------------------------+-----------------------------------------------+-----+
|109487 |Interstellar (2014)                                      |Sci-Fi|IMAX                                    |21038|
|79132  |Inception (2010)                                         |Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX|12824|
|318    |Shawshank Redemption, The (1994)                         |Crime|Drama                                    |10247|
|58559  |Dark Knight, The (2008)                                  |Action|Crime|Drama|IMAX                        |10224|
|2571   |Matrix, The (1999)                                       |Action|Sci-Fi|Thriller                         |9867 |
|2959   |Fight Club (199

In [31]:
movies.filter(
    col("title").contains("Inception")
).show(truncate=False)

+-------+----------------+-----------------------------------------------+
|movieId|title           |genres                                         |
+-------+----------------+-----------------------------------------------+
|79132  |Inception (2010)|Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX|
+-------+----------------+-----------------------------------------------+



In [37]:
inception_fans = ratings.filter( 
    (col("movieId") == 79132) & (col("rating") >= 4.5)
)

inception_fans.show(10)
print(inception_fans.count())

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     3|  79132|   5.0|1536173845|
|    14|  79132|   4.5|1311532977|
|    15|  79132|   4.5|1463468340|
|    23|  79132|   5.0|1514249446|
|    33|  79132|   4.5|1587178704|
|    57|  79132|   5.0|1622601641|
|    87|  79132|   5.0|1442832541|
|   106|  79132|   4.5|1459798653|
|   108|  79132|   5.0|1295749438|
|   112|  79132|   4.5|1477464634|
+------+-------+------+----------+
only showing top 10 rows


33778


In [39]:
fan_user_ids = inception_fans.select("userId").distinct()

similar_taste_movies = (
    ratings
    .join(fan_user_ids, on="userId")
    .filter((col("rating") >= 4.5) & (col("movieId") != 79132))
    .join(movies, on="movieId")
    .groupBy("movieId", "title", "genres")
    .count()
    .orderBy(col("count").desc())
)

similar_taste_movies.show(20, truncate=False)

+-------+---------------------------------------------------------+------------------------------+-----+
|movieId|title                                                    |genres                        |count|
+-------+---------------------------------------------------------+------------------------------+-----+
|318    |Shawshank Redemption, The (1994)                         |Crime|Drama                   |16754|
|58559  |Dark Knight, The (2008)                                  |Action|Crime|Drama|IMAX       |16592|
|2571   |Matrix, The (1999)                                       |Action|Sci-Fi|Thriller        |16405|
|2959   |Fight Club (1999)                                        |Action|Crime|Drama|Thriller   |15503|
|109487 |Interstellar (2014)                                      |Sci-Fi|IMAX                   |12824|
|7153   |Lord of the Rings: The Return of the King, The (2003)    |Action|Adventure|Drama|Fantasy|12785|
|4993   |Lord of the Rings: The Fellowship of the Ring,

In [45]:
toy_story_movies = movies.filter(
    col("title").contains("Toy Story")
)

toy_story_movies.show(truncate=False)

+-------+-----------------------------------------+------------------------------------------------+
|movieId|title                                    |genres                                          |
+-------+-----------------------------------------+------------------------------------------------+
|1      |Toy Story (1995)                         |Adventure|Animation|Children|Comedy|Fantasy     |
|3114   |Toy Story 2 (1999)                       |Adventure|Animation|Children|Comedy|Fantasy     |
|78499  |Toy Story 3 (2010)                       |Adventure|Animation|Children|Comedy|Fantasy|IMAX|
|106022 |Toy Story of Terror (2013)               |Animation|Children|Comedy                       |
|115875 |Toy Story Toons: Hawaiian Vacation (2011)|Adventure|Animation|Children|Comedy|Fantasy     |
|115879 |Toy Story Toons: Small Fry (2011)        |Adventure|Animation|Children|Comedy|Fantasy     |
|120468 |Toy Story Toons: Partysaurus Rex (2012)  |Animation|Children|Comedy               

In [46]:
toy_story_ids = [
    row["movieId"]
    for row in toy_story_movies.select("movieId").collect()
]

In [47]:
toy_story_fans = ratings.filter(
    (col("movieId").isin(toy_story_ids))
    & (col("rating") >= 4.5)
)


toy_story_fans.show(10)
print(toy_story_fans.count())

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     2|      1|   5.0| 835815971|
|     3|   3114|   5.0|1536174265|
|    12|      1|   5.0| 862500738|
|    24|      1|   4.5|1062943926|
|    24|   3114|   4.5|1062943613|
|    35|   3114|   4.5|1119980870|
|    40|   3114|   4.5|1452041689|
|    51|      1|   5.0| 862602100|
|    64|      1|   5.0|1111104291|
|    72|   3114|   4.5|1268751003|
+------+-------+------+----------+
only showing top 10 rows


44311


In [49]:
fan_user_ids = toy_story_fans.select("userId").distinct()

similar_taste_movies = (
    ratings
    .join(fan_user_ids, on="userId")
    .filter((col("rating") >= 4.5) & (~col("movieId").isin(toy_story_ids)))
    .join(movies, on="movieId")
    .groupBy("movieId", "title", "genres")
    .count()
    .orderBy(col("count").desc())
)

similar_taste_movies.show(20, truncate=False)

+-------+------------------------------------------------------------------------------+-----------------------------------------------+-----+
|movieId|title                                                                         |genres                                         |count|
+-------+------------------------------------------------------------------------------+-----------------------------------------------+-----+
|318    |Shawshank Redemption, The (1994)                                              |Crime|Drama                                    |13412|
|260    |Star Wars: Episode IV - A New Hope (1977)                                     |Action|Adventure|Sci-Fi                        |11766|
|2571   |Matrix, The (1999)                                                            |Action|Sci-Fi|Thriller                         |10916|
|356    |Forrest Gump (1994)                                                           |Comedy|Drama|Romance|War                       |10895|